<a name="top"></a><img src="images/chisel_1024.png" alt="Chisel logo" style="width:480px;" />

# 模块 3.1：生成器：参数
**上一步：[ChiselTest (曾用名 chisel-testers2)](2.6_chiseltest.ipynb)**<br>
**下一步：[生成器：集合](3.2_collections.ipynb)**

## 动机
对于 Chisel 模块作为代码生成器，必须有某种东西告诉生成器它应该如何工作。
在本节中，我们讨论模块参数化、各种方法论和 Scala 语言特性。
参数传递实现的丰富性与生成的电路的丰富性成正比。
参数应该提供有用的默认值，易于设置，并防止非法或无意义的值。
对于更复杂的系统，如果它们可以以不无意中影响其他模块使用的方式在本地被覆盖，则非常有用。

## 设置

In [ ]:
val path = System.getProperty("user.dir") + "/source/load-ivy.sc"
interp.load.module(ammonite.ops.Path(java.nio.file.FileSystems.getDefault().getPath(path)))

In [ ]:
import chisel3._
import chisel3.util._
import chisel3.tester._
import chisel3.tester.RawTester.test

---
# 参数传递
Chisel 提供了强大的结构来编写硬件生成器。
生成器是接受一些电路参数并产生电路描述的程序。
在本节中，我们将首先讨论 Chisel 生成器如何获取其参数。

<span style="color:blue">**示例：参数化 Scala 对象**</span><br>
每个 Chisel `Module` 都是一个 Scala 类，就像任何其他类一样。
回想一下，Scala 类可以像这样参数化：

In [ ]:
class ParameterizedScalaObject(param1: Int, param2: String) {
  println(s"我有参数：param1 = $param1 和 param2 = $param2")
}
val obj1 = new ParameterizedScalaObject(4,     "Hello")
val obj2 = new ParameterizedScalaObject(4 + 2, "World")

<span style="color:blue">**示例：参数化 Chisel 对象**</span><br>
Chisel 模块可以以相同的方式参数化。
以下模块具有其所有输入和输出宽度的参数。
运行代码块将打印生成的 Verilog。
尝试使用参数并检查输出是否更改以反映新参数。

In [ ]:
class ParameterizedWidthAdder(in0Width: Int, in1Width: Int, sumWidth: Int) extends Module {
  require(in0Width >= 0)
  require(in1Width >= 0)
  require(sumWidth >= 0)
  val io = IO(new Bundle {
    val in0 = Input(UInt(in0Width.W))
    val in1 = Input(UInt(in1Width.W))
    val sum = Output(UInt(sumWidth.W))
  })
  // a +& b 包含进位，a + b 不包含
  io.sum := io.in0 +& io.in1
}

println(getVerilog(new ParameterizedWidthAdder(1, 4, 6)))

上面的代码块有一些 `require(...)` 语句。
这些是预细化断言，当您的生成器仅适用于某些参数化或某些参数化互斥或无意义时，这些断言非常有用。
上面的代码块检查宽度是否为非负数。

有一个单独的结构用于仿真时断言，称为 `assert(...)`。

## 使用参数化模块进行排序
以下代码块是一个参数化排序，类似于模块 2.3 中的 `Sort4`。
与前面具有参数化宽度 IO 的加法器示例不同，此示例具有固定的 IO。
该参数控制模块内部生成的硬件。
![Sort4](images/Sorter4.png)
<span style="color:blue">**示例：参数化 4 输入排序**</span><br>
与 2.3 不同，此实现被参数化为降序或升序排序。

In [ ]:
/** Sort4 将其 4 个输入排序到其 4 个输出 */
class Sort4(ascending: Boolean) extends Module {
  val io = IO(new Bundle {
    val in0 = Input(UInt(16.W))
    val in1 = Input(UInt(16.W))
    val in2 = Input(UInt(16.W))
    val in3 = Input(UInt(16.W))
    val out0 = Output(UInt(16.W))
    val out1 = Output(UInt(16.W))
    val out2 = Output(UInt(16.W))
    val out3 = Output(UInt(16.W))
  })
    
  // 此比较函数根据模块的参数化决定 < 或 >
  def comp(l: UInt, r: UInt): Bool = {
      if (ascending) {
        l < r
      } else {
        l > r
    }
  }

  val row10 = Wire(UInt(16.W))
  val row11 = Wire(UInt(16.W))
  val row12 = Wire(UInt(16.W))
  val row13 = Wire(UInt(16.W))

  when(comp(io.in0, io.in1)) {
    row10 := io.in0            // 保留前两个元素
    row11 := io.in1
  }.otherwise {
    row10 := io.in1            // 交换前两个元素
    row11 := io.in0
  }

  when(comp(io.in2, io.in3)) {
    row12 := io.in2            // 保留后两个元素
    row13 := io.in3
  }.otherwise {
    row12 := io.in3            // 交换后两个元素
    row13 := io.in2
  }

  val row21 = Wire(UInt(16.W))
  val row22 = Wire(UInt(16.W))

  when(comp(row11, row12)) {
    row21 := row11            // 保留中间两个元素
    row22 := row12
  }.otherwise {
    row21 := row12            // 交换中间两个元素
    row22 := row11
  }

  val row20 = Wire(UInt(16.W))
  val row23 = Wire(UInt(16.W))
  when(comp(row10, row13)) {
    row20 := row10            // 保留第一个和第四个元素
    row23 := row13
  }.otherwise {
    row20 := row13            // 交换第一个和第四个元素
    row23 := row10
  }

  when(comp(row20, row21)) {
    io.out0 := row20            // 保留前两个元素
    io.out1 := row21
  }.otherwise {
    io.out0 := row21            // 交换前两个元素
    io.out1 := row20
  }

  when(comp(row22, row23)) {
    io.out2 := row22            // 保留前两个元素
    io.out3 := row23
  }.otherwise {
    io.out2 := row23            // 交换前两个元素
    io.out3 := row22
  }
}



// 这是测试程序
test(new Sort4(true)) { c => 
  c.io.in0.poke(3.U)
  c.io.in1.poke(6.U)
  c.io.in2.poke(9.U)
  c.io.in3.poke(12.U)
  c.io.out0.expect(3.U)
  c.io.out1.expect(6.U)
  c.io.out2.expect(9.U)
  c.io.out3.expect(12.U)

  c.io.in0.poke(13.U)
  c.io.in1.poke(4.U)
  c.io.in2.poke(6.U)
  c.io.in3.poke(1.U)
  c.io.out0.expect(1.U)
  c.io.out1.expect(4.U)
  c.io.out2.expect(6.U)
  c.io.out3.expect(13.U)

  c.io.in0.poke(13.U)
  c.io.in1.poke(6.U)
  c.io.in2.poke(4.U)
  c.io.in3.poke(1.U)
  c.io.out0.expect(1.U)
  c.io.out1.expect(4.U)
  c.io.out2.expect(6.U)
  c.io.out3.expect(13.U)
}
test(new Sort4(false)) { c =>
  c.io.in0.poke(3.U)
  c.io.in1.poke(6.U)
  c.io.in2.poke(9.U)
  c.io.in3.poke(12.U)
  c.io.out0.expect(12.U)
  c.io.out1.expect(9.U)
  c.io.out2.expect(6.U)
  c.io.out3.expect(3.U)

  c.io.in0.poke(13.U)
  c.io.in1.poke(4.U)
  c.io.in2.poke(6.U)
  c.io.in3.poke(1.U)
  c.io.out0.expect(13.U)
  c.io.out1.expect(6.U)
  c.io.out2.expect(4.U)
  c.io.out3.expect(1.U)

  c.io.in0.poke(1.U)
  c.io.in1.poke(6.U)
  c.io.in2.poke(4.U)
  c.io.in3.poke(13.U)
  c.io.out0.expect(13.U)
  c.io.out1.expect(6.U)
  c.io.out2.expect(4.U)
  c.io.out3.expect(1.U)
}
println("成功！！") // Scala 代码：如果我们到达这里，我们的测试通过了！

---
# Option 和默认参数

有时函数会返回值，有时则不会。Scala 有一种机制可以在类型系统中对此进行编码，而不是在无法返回值时出错。

<span style="color:blue">**示例：错误的 Map 索引调用**</span><br>
在以下示例中，我们有一个包含多个键/值对的映射。如果我们尝试访问一个不存在的键/值对，则会发生运行时错误：

In [ ]:
val map = Map("a" -> 1)
val a = map("a")
println(a)
val b = map("b")
println(b)

<span style="color:blue">**示例：获取不确定的索引**</span><br>
然而，`Map` 提供了另一种访问键值的方法，即通过 **get** 方法。使用此方法返回一个抽象类 `Option` 的值。`Option` 有两个子类，`Some` 和 `None`。

In [ ]:
val map = Map("a" -> 1)
val a = map.get("a")
println(a)
val b = map.get("b")
println(b)

正如您将在后续章节中看到的，`Option` 非常重要，因为它允许用户使用 match 语句来检查 Scala 类型和值。

<span style="color:blue">**示例：Get Or Else！**</span><br>
与 `Map` 类似，`Option` 也有一个 `get` 方法，如果在 `None` 上调用该方法，则会出错。对于这些情况，我们可以使用 **`getOrElse`** 提供一个默认值。

In [ ]:
val some = Some(1)
val none = None
println(some.get)          // 返回 1
// println(none.get)       // 错误！
println(some.getOrElse(2)) // 返回 1
println(none.getOrElse(2)) // 返回 2

## 具有默认值的参数选项

当对象或函数有很多参数时，一直完整地指定它们可能会很繁琐且容易出错。
在模块 1 中，您学习了命名参数和参数默认值。
有时，参数没有一个好的默认值。
在这些情况下，可以使用 `Option` 并将默认值设置为 `None`。

<span style="color:blue">**示例：可选复位**</span><br>
以下代码块显示了一个将其输入延迟一个时钟周期的模块。
如果 `resetValue = None`（这是默认值），则寄存器将没有复位值，并将初始化为垃圾值。
这避免了使用超出正常范围的值来表示“无”（例如使用 -1 作为复位值来表示此寄存器未复位）这种常见但不美观的情况。

In [ ]:
class DelayBy1(resetValue: Option[UInt] = None) extends Module {
    val io = IO(new Bundle {
        val in  = Input( UInt(16.W))
        val out = Output(UInt(16.W))
    })
    val reg = if (resetValue.isDefined) { // resetValue = Some(数字)
        RegInit(resetValue.get)
    } else { //resetValue = None
        Reg(UInt())
    }
    reg := io.in
    io.out := reg
}

println(getVerilog(new DelayBy1))
println(getVerilog(new DelayBy1(Some(3.U))))

---
# Match/Case 语句
Scala 的 *matching* 概念在整个 Chisel 中都有使用，并且需要成为任何 Chisel 程序员基本理解的一部分。Scala 提供了 match 运算符，它支持：
- 简单的备选方案测试，类似于 C 语言的 *switch* 语句
- 更复杂的值的临时组合测试
- 当变量的类型未知或未指定时，根据变量的类型采取操作，例如当
  - 变量取自异构列表 ```val mixedList = List(1, "string", false)```
  - 或者变量已知是超类的成员，但不知道它是哪个特定的子类。
- 提取使用*正则表达式*指定的字符串的子字符串


<span style="color:blue">**示例：值匹配**</span><br>
在以下示例中，根据我们 **match** 的变量的 **value**，我们执行不同的 **case** 语句：

In [ ]:
// y 是在代码中其他地方定义的整数变量
val y = 7
/// ...
val x = y match {
  case 0 => "zero" // 一种常见的语法，如果适合一行则首选
  case 1 =>        // 另一种常见的语法，如果不适合一行则首选。
      "one"        // 注意代码块一直持续到下一个 case
  case 2 => {      // 另一种语法，但花括号不是必需的
      "two"
  }
  case _ => "many" // _ 是一个匹配所有值的通配符
}
println("y is " + x)

match 运算符检查可能的值，并为每种情况返回一个字符串。需要注意以下几点：
- 每个跟在 ```=>``` 运算符后面的代码块都会一直持续到遇到 match 的结束大括号或下一个 case 语句。
- match 会按照 case 语句的顺序进行搜索，一旦某个 case 语句匹配成功，就不会再对其他 case 语句进行检查。
- 使用下划线作为通配符，用于处理任何未找到的值。

<span style="color:blue">**示例：多值匹配**</span><br>
此外，可以同时匹配多个变量。这是一个使用 match 语句和值元组实现的真值表的简单示例：

In [ ]:
def animalType(biggerThanBreadBox: Boolean, meanAsCanBe: Boolean): String = {
  (biggerThanBreadBox, meanAsCanBe) match {
    case (true, true) => "wolverine"
    case (true, false) => "elephant"
    case (false, true) => "shrew"
    case (false, false) => "puppy"
  }
}
println(animalType(true, true))

<span style="color:blue">**示例：类型匹配**</span><br>
Scala 是一种强类型语言，因此所有对象的类型在运行时都是已知的。我们可以使用 **match 语句** 来利用此类型信息来决定控制流：

In [ ]:
val sequence = Seq("a", 1, 0.0)
sequence.foreach { x =>
  x match {
    case s: String => println(s"$x 是一个字符串")
    case s: Int    => println(s"$x 是一个整数")
    case s: Double => println(s"$x 是一个双精度浮点数")
    case _ => println(s"$x 是一个未知类型！")
  }
}

<span style="color:blue">**示例：多类型匹配**</span><br>
如果要匹配一个值是否属于多种类型之一，请使用以下语法。*请注意，匹配时**必须**使用 `_`。*

In [ ]:
val sequence = Seq("a", 1, 0.0)
sequence.foreach { x =>
  x match {
    case _: Int | _: Double => println(s"$x 是一个数字！")
    case _ => println(s"$x 是一个未知类型！")
  }
}

<span style="color:blue">**示例：类型匹配和擦除**</span><br>
类型匹配有一些限制。因为 Scala 在 JVM 上运行，而 JVM 不维护多态类型，所以您无法在运行时匹配它们（因为它们都被擦除了）。请注意，以下示例始终匹配第一个 case 语句，因为 `[String]`、`[Int]` 和 `[Double]` 多态类型被擦除，并且 case 语句**实际上**只匹配一个 `Seq`。

In [ ]:
val sequence = Seq(Seq("a"), Seq(1), Seq(0.0))
sequence.foreach { x =>
  x match {
    case s: Seq[String] => println(s"$x 是一个字符串")
    case s: Seq[Int]    => println(s"$x 是一个整数")
    case s: Seq[Double] => println(s"$x 是一个双精度浮点数")
  }
}

请注意，如果您实现类似上述示例的代码，Scala 编译器通常会给出警告。

<span style="color:blue">**示例：可选复位匹配**</span><br>
以下代码块显示了使用 match 结构而不是 `if/else` 的相同 `DelayBy1` 模块。

In [ ]:
class DelayBy1(resetValue: Option[UInt] = None) extends Module {
  val io = IO(new Bundle {
    val in  = Input( UInt(16.W))
    val out = Output(UInt(16.W))
  })
  val reg = resetValue match {
    case Some(r) => RegInit(r)
    case None    => Reg(UInt())
  }
  reg := io.in
  io.out := reg
}

println(getVerilog(new DelayBy1))
println(getVerilog(new DelayBy1(Some(3.U))))

---
# 具有可选字段的 IO

有时我们希望 IO 可以选择性地包含或排除。
也许有一些内部状态对于调试来说很好查看，但是当生成器在系统中使用时，您希望隐藏它。
也许您的生成器有一些输入，在并非所有情况下都需要连接，因为有一个合理的默认值。

<span style="color:blue">**示例：使用 Option 的可选 IO**</span><br>
可选的 bundle 字段是获得此功能的一种方法。
在以下示例中，我们展示了一个一位加法器，它可以选择性地接受进位。
如果包含进位，`io.carryIn` 将具有 `Some[UInt]` 类型并包含在 IO bundle 中。
如果不包含进位，`io.carryIn` 将具有 `None` 类型并将从 IO bundle 中排除。

In [ ]:
class HalfFullAdder(val hasCarry: Boolean) extends Module {
  val io = IO(new Bundle {
    val a = Input(UInt(1.W))
    val b = Input(UInt(1.W))
    val carryIn = if (hasCarry) Some(Input(UInt(1.W))) else None
    val s = Output(UInt(1.W))
    val carryOut = Output(UInt(1.W))
  })
  val sum = io.a +& io.b +& io.carryIn.getOrElse(0.U)
  io.s := sum(0)
  io.carryOut := sum(1)
}

test(new HalfFullAdder(false)) { c =>
  require(!c.hasCarry, "DUT 必须是半加器")
  // 0 + 0 = 0
  c.io.a.poke(0.U)
  c.io.b.poke(0.U)
  c.io.s.expect(0.U)
  c.io.carryOut.expect(0.U)
  // 0 + 1 = 1
  c.io.b.poke(1.U)
  c.io.s.expect(1.U)
  c.io.carryOut.expect(0.U)
  // 1 + 1 = 2
  c.io.a.poke(1.U)
  c.io.s.expect(0.U)
  c.io.carryOut.expect(1.U)
  // 1 + 0 = 1
  c.io.b.poke(0.U)
  c.io.s.expect(1.U)
  c.io.carryOut.expect(0.U)
}

test(new HalfFullAdder(true)) { c =>
  require(c.hasCarry, "DUT 必须是半加器")
  c.io.carryIn.get.poke(0.U)
  // 0 + 0 + 0 = 0
  c.io.a.poke(0.U)
  c.io.b.poke(0.U)
  c.io.s.expect(0.U)
  c.io.carryOut.expect(0.U)
  // 0 + 0 + 1 = 1
  c.io.b.poke(1.U)
  c.io.s.expect(1.U)
  c.io.carryOut.expect(0.U)
  // 0 + 1 + 1 = 2
  c.io.a.poke(1.U)
  c.io.s.expect(0.U)
  c.io.carryOut.expect(1.U)
  // 0 + 1 + 0 = 1
  c.io.b.poke(0.U)
  c.io.s.expect(1.U)
  c.io.carryOut.expect(0.U)

  c.io.carryIn.get.poke(1.U)
  // 1 + 0 + 0 = 1
  c.io.a.poke(0.U)
  c.io.b.poke(0.U)
  c.io.s.expect(1.U)
  c.io.carryOut.expect(0.U)
  // 1 + 0 + 1 = 2
  c.io.b.poke(1.U)
  c.io.s.expect(0.U)
  c.io.carryOut.expect(1.U)
  // 1 + 1 + 1 = 3
  c.io.a.poke(1.U)
  c.io.s.expect(1.U)
  c.io.carryOut.expect(1.U)
  // 1 + 1 + 0 = 2
  c.io.b.poke(0.U)
  c.io.s.expect(0.U)
  c.io.carryOut.expect(1.U)
}

println("成功！！") // Scala 代码：如果我们到达这里，我们的测试通过了！

<span style="color:blue">**示例：使用零宽度线的可选 IO**</span><br>
另一种实现与 `Option` 类似功能的方法是使用零宽度线。
Chisel 类型允许宽度为零。
宽度为零的 IO 会从生成的 Verilog 中删除，任何尝试使用零宽度线值的内容都会得到一个常量零。
如果零是一个合理的默认值，那么零宽度线会很好，因为它们避免了对选项进行匹配或调用 `getOrElse` 的需要。

In [ ]:
class HalfFullAdder(val hasCarry: Boolean) extends Module {
  val io = IO(new Bundle {
    val a = Input(UInt(1.W))
    val b = Input(UInt(1.W))
    val carryIn = Input(if (hasCarry) UInt(1.W) else UInt(0.W))
    val s = Output(UInt(1.W))
    val carryOut = Output(UInt(1.W))
  })
  val sum = io.a +& io.b +& io.carryIn
  io.s := sum(0)
  io.carryOut := sum(1)
}
println("半加器：")
println(getVerilog(new HalfFullAdder(false)))
println("\n\n全加器：")
println(getVerilog(new HalfFullAdder(true)))

---
# Implicits
在编程时，经常需要大量样板代码。为了处理这种情况，Scala 引入了 **implicits** 的概念，它允许编译器为您执行一些语法糖。由于许多事情在幕后发生，implicits 可能看起来非常神奇。本节分解了一些基本示例来解释它们是什么以及它们通常在哪里使用。

## 隐式参数
有时，您的代码需要从一系列函数调用的深处访问某个顶级变量。您可以使用隐式参数来代替手动将此变量传递给每个函数调用。

<span style="color:blue">**示例：隐式猫**</span><br>
在以下示例中，我们可以隐式或显式传递猫的数量。

In [ ]:
object CatDog {
  implicit val numberOfCats: Int = 3
  //implicit val numberOfDogs: Int = 5

  def tooManyCats(nDogs: Int)(implicit nCats: Int): Boolean = nCats > nDogs
    
  val imp = tooManyCats(2)    // 参数隐式传递！
  val exp = tooManyCats(2)(1) // 参数显式传递！
}
CatDog.imp
CatDog.exp

这里发生了什么？首先，我们定义了一个隐式值 **numberOfCats**。在给定的作用域中，**对于给定的类型只能有一个隐式值**。然后，我们定义了一个接受两个参数列表的函数；第一个是任何显式参数，第二个是任何隐式参数。当我们调用 **tooManyCats** 时，我们要么省略第二个隐式参数列表（让编译器为我们找到它），要么显式提供一个参数（它可以不同于隐式值）。

以下是隐式参数可能*失败*的方式：
- 在一个作用域中定义了两个或多个给定类型的隐式值
- 如果编译器找不到函数调用所需的隐式值

<span style="color:blue">**示例：隐式日志记录**</span><br>
下一个代码块显示了如何在 Chisel 生成器中使用隐式参数来实现日志记录。

***注意：在 Scala 中有更好的日志记录方法！***

In [ ]:
sealed trait Verbosity
implicit case object Silent extends Verbosity
case object Verbose extends Verbosity

class ParameterizedWidthAdder(in0Width: Int, in1Width: Int, sumWidth: Int)(implicit verbosity: Verbosity)
extends Module {
  def log(msg: => String): Unit = verbosity match {
    case Silent =>
    case Verbose => println(msg)
  }
  require(in0Width >= 0)
  log(s"in0Width of $in0Width OK")
  require(in1Width >= 0)
  log(s"in1Width of $in1Width OK")
  require(sumWidth >= 0)
  log(s"sumWidth of $sumWidth OK")
  val io = IO(new Bundle {
    val in0 = Input(UInt(in0Width.W))
    val in1 = Input(UInt(in1Width.W))
    val sum = Output(UInt(sumWidth.W))
  })
  log("Made IO")
  io.sum := io.in0 + io.in1
  log("Assigned output")
}

println(getVerilog(new ParameterizedWidthAdder(1, 4, 5)))
println(getVerilog(new ParameterizedWidthAdder(1, 4, 5)(Verbose)))

## 隐式转换
与隐式参数类似，隐式函数（也称为**隐式转换**）用于减少样板代码。更具体地说，它们用于自动将一个 Scala 对象转换为另一个 Scala 对象。

<span style="color:blue">**示例：隐式转换**</span><br>
在以下示例中，我们有两个类，`Animal` 和 `Human`。`Animal` 有一个 `species` 字段，但 `Human` 没有。但是，通过实现隐式转换，我们可以在 `Human` 上调用 `species`。

In [ ]:
class Animal(val name: String, val species: String)
class Human(val name: String)
implicit def human2animal(h: Human): Animal = new Animal(h.name, "Homo sapiens")
val me = new Human("Adam")
println(me.species)

通常，隐式转换会使您的代码难以理解，因此我们建议您将其作为最后的手段。首先尝试继承、特质或方法重载。

---
# 生成器示例
以下示例显示了一个 1 位输入 Mealy 机的生成器。
它有一个基于 [Wikipedia](https://en.wikipedia.org/wiki/Mealy_machine#/media/File:Mealy.png) 示例的测试。
通读代码并尝试理解其工作原理。

<span style="color:blue">**示例：Mealy 机**</span><br>
尝试在下面的代码块中创建您自己的 Mealy 机生成器参数化并编写您自己的测试。

In [ ]:
// Mealy 机具有
case class BinaryMealyParams(
  // 状态数
  nStates: Int,
  // 初始状态
  s0: Int,
  // 描述状态转换的函数
  stateTransition: (Int, Boolean) => Int,
  // 描述输出的函数
  output: (Int, Boolean) => Int
) {
  require(nStates >= 0)
  require(s0 < nStates && s0 >= 0)
}

class BinaryMealy(val mp: BinaryMealyParams) extends Module {
  val io = IO(new Bundle {
    val in = Input(Bool())
    val out = Output(UInt())
  })

  val state = RegInit(UInt(), mp.s0.U)

  // 如果没有状态，则输出零
  io.out := 0.U
  for (i <- 0 until mp.nStates) {
    when (state === i.U) {
      when (io.in) {
        state  := mp.stateTransition(i, true).U
        io.out := mp.output(i, true).U
      }.otherwise {
        state  := mp.stateTransition(i, false).U
        io.out := mp.output(i, false).U
      }
    }
  }
}

// 来自 https://en.wikipedia.org/wiki/Mealy_machine 的示例
val nStates = 3
val s0 = 2
def stateTransition(state: Int, in: Boolean): Int = {
  if (in) {
    1
  } else {
    0
  }
}
def output(state: Int, in: Boolean): Int = {
  if (state == 2) {
    return 0
  }
  if ((state == 1 && !in) || (state == 0 && in)) {
    return 1
  } else {
    return 0
  }
}

val testParams = BinaryMealyParams(nStates, s0, stateTransition, output)

test(new BinaryMealy(testParams)) { c =>
  c.io.in.poke(false.B)
  c.io.out.expect(0.U)
  c.clock.step(1)
  c.io.in.poke(false.B)
  c.io.out.expect(0.U)
  c.clock.step(1)
  c.io.in.poke(false.B)
  c.io.out.expect(0.U)
  c.clock.step(1)
  c.io.in.poke(true.B)
  c.io.out.expect(1.U)
  c.clock.step(1)
  c.io.in.poke(true.B)
  c.io.out.expect(0.U)
  c.clock.step(1)
  c.io.in.poke(false.B)
  c.io.out.expect(1.U)
  c.clock.step(1)
  c.io.in.poke(true.B)
  c.io.out.expect(1.U)
  c.clock.step(1)
  c.io.in.poke(false.B)
  c.io.out.expect(1.U)
  c.clock.step(1)
  c.io.in.poke(true.B)
  c.io.out.expect(1.U)
}

println("成功！！") // Scala 代码：如果我们到达这里，我们的测试通过了！

---
# 您已完成！

[返回顶部。](#top)